In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np

files = {
    "0h": "/Users/apple/Downloads/bacdrops/cip/0h.matrix.tsv",
    "1h": "/Users/apple/Downloads/bacdrops/cip/1h.matrix.tsv",
    "2h": "/Users/apple/Downloads/bacdrops/cip/2h.matrix.tsv",
    "4h": "/Users/apple/Downloads/bacdrops/cip/4h.matrix.tsv",
}
adata_list = []
import re
ncrna_genes = set()
remove_features = {"rRNA", 'tRNA'}
with open("eco.gff") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if len(cols) != 9:
            continue
        if cols[2] not in remove_features:
            continue
        m = re.search(r"Parent=([^;]+)", cols[8])
        if m:
            ncrna_genes.add(m.group(1))
for tp, fn in files.items():
    df = pd.read_csv(fn, sep="\t", index_col=0)
    ad = sc.AnnData(df.T)
    ad.obs_names = [f"{tp}_{x}" for x in ad.obs_names]
    ad.obs["time"] = tp
    # QC
    ad.obs["total_umis"] = np.asarray(ad.X.sum(axis=1)).ravel()
    ad.obs["n_genes"] = np.asarray((ad.X > 0).sum(axis=1)).ravel()
    ncrna_mask = ad.var_names.isin(ncrna_genes)
    ncrna_counts = np.asarray(ad[:, ncrna_mask].X.sum(axis=1)).ravel()
    ad.obs["ncrna_pct"] = ncrna_counts / ad.obs["total_umis"] * 100
    ad = ad[:, ~ncrna_mask].copy()
    print(f"{tp}: {ncrna_mask.sum()} ncRNA genes removed, " f"{ad.shape[1]} genes remaining")
    adata_list.append(ad)

adata = sc.concat(adata_list, join="outer", label="batch", keys=list(files.keys()), fill_value=0)
print(adata)

In [ ]:
import scanpy as sc
adata = sc.read_h5ad("/Users/apple/Downloads/cip/CIP_new.h5ad")

In [ ]:
# Normalize
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)
adata_full = adata.copy()

sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor="seurat")
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
# PCA
sc.tl.pca(adata_hvg, n_comps=30)
# neighbors
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30)
# UMAP
sc.tl.umap(adata_hvg)
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
sc.pl.umap(adata_hvg, color="time", size=10, ax=ax, show=False, frameon=True)
for collection in ax.collections:
    collection.set_rasterized(True)
plt.show()
#note4fig: fig3a_part2

In [ ]:
import pandas as pd
import re

gene_map = []
with open("eco.gff") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if len(cols) != 9:
            continue
        if cols[2] != "gene":
            continue
        attrs = cols[8]
        gene_id = None
        gene_name = None
        m = re.search(r"ID=([^;]+)", attrs)
        if m:
            gene_id = m.group(1)
        m = re.search(r"gene=([^;]+)", attrs)
        if m:
            gene_name = m.group(1)
        else:
            m = re.search(r"Name=([^;]+)", attrs)
            if m:
                gene_name = m.group(1)
        if gene_id is not None:
            gene_map.append([gene_id, gene_name])
gene_map = pd.DataFrame(gene_map, columns=["names", "gene_name"])
print(gene_map.head())

In [ ]:
gene_dict = dict(zip(gene_map["names"], gene_map["gene_name"]))
adata.var["gene_id"] = adata.var_names
adata.var["gene_name"] = [gene_dict.get(x, x) for x in adata.var_names]
adata.var_names = adata.var["gene_name"]
adata.var_names_make_unique()

In [ ]:
import numpy as np
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
# Connectivity check
n_comp, labels = connected_components(adata_hvg.obsp["connectivities"], directed=False)
adata_hvg.obs["graph_component"] = labels.astype(str)
print(f"Graph components: {n_comp}")
# Root = 0h (untreated)
root_time = "0h"
root_candidates = np.flatnonzero(adata_hvg.obs["time"].astype(str).eq(root_time).to_numpy())
pc_centroid = adata_hvg.obsm["X_pca"][root_candidates].mean(axis=0)
root_idx = root_candidates[np.argmin(((adata_hvg.obsm["X_pca"][root_candidates] - pc_centroid) ** 2).sum(axis=1))]
adata_hvg.uns["iroot"] = int(root_idx)
print(f"DPT root: {adata_hvg.obs_names[root_idx]} (time={root_time})")
# DPT
sc.tl.diffmap(adata_hvg, n_comps=15, random_state=42)
sc.tl.dpt(adata_hvg, n_dcs=10)
adata_hvg.obs["pseudotime"] = adata_hvg.obs["dpt_pseudotime"].replace([np.inf, -np.inf], np.nan)
print(f"Finite pseudotime: {adata_hvg.obs['pseudotime'].notna().sum()}")

In [ ]:
# UMAP: time (left) + pseudotime trajectory (right)
# Same coordinates as original UMAP
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
TIME_ORDER = ["0h","1h","2h","4h"]
TIME_COLORS = {"0h":"#440154","1h":"#31688e","2h":"#35b779","4h":"#fde725"}
fig, axes = plt.subplots(1, 2, figsize=(11, 5), dpi=300)
c = pd.DataFrame(
    adata_hvg.obsm["X_umap"],
    index=adata_hvg.obs_names,
    columns=["UMAP1","UMAP2"]
).join(adata_hvg.obs[["time","pseudotime"]])
# Left: UMAP by sampling time (same as original)
for time in TIME_ORDER:
    d = c[c.time.astype(str).eq(time)]
    axes[0].scatter(d.UMAP1, d.UMAP2, s=10, c=TIME_COLORS[time], alpha=.35, label=time, rasterized=True)
axes[0].set(title="UMAP by sampling time", xlabel="UMAP1", ylabel="UMAP2")
axes[0].legend(frameon=False, markerscale=2)
# Right: UMAP by pseudotime with trajectory arrows
valid = c.pseudotime.notna()
axes[1].scatter(c.loc[~valid, "UMAP1"], c.loc[~valid, "UMAP2"], s=10, c="#d9d9d9", alpha=.25, rasterized=True)
sc_plot = axes[1].scatter(c.loc[valid, "UMAP1"], c.loc[valid, "UMAP2"], s=10, c=c.loc[valid, "pseudotime"], cmap="viridis", alpha=.5, rasterized=True)
d = c.loc[valid].copy()
d["bin"] = pd.qcut(d.pseudotime.rank(method="first"), 12, labels=False)
centers = d.groupby("bin", observed=True)[["UMAP1","UMAP2","pseudotime"]].median().sort_values("pseudotime")
axes[1].plot(centers.UMAP1, centers.UMAP2, c="black", lw=2.2, zorder=5)
for i in range(len(centers)-1):
    axes[1].annotate("", xy=centers.iloc[i+1][["UMAP1","UMAP2"]], xytext=centers.iloc[i][["UMAP1","UMAP2"]],
                     arrowprops={"arrowstyle":"->","color":"black","lw":1.7}, zorder=6)
r = c.iloc[root_idx]
axes[1].scatter(r.UMAP1, r.UMAP2, marker="*", s=200, c="#f9c74f", edgecolor="black", zorder=7)
axes[1].set(title="DPT trajectory from 0h root", xlabel="UMAP1", ylabel="UMAP2")
plt.colorbar(sc_plot, ax=axes[1], shrink=0.6, label="pseudotime")
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3c_part2 (panel2 was extracted and used)

In [ ]:
import seaborn as sns
valid = adata_hvg.obs["pseudotime"].notna().to_numpy()
gene_ids = adata_hvg.var_names.to_numpy()
gene_names = np.array([gene_dict.get(gid, gid) for gid in gene_ids])
expr = adata_hvg[valid].X
if hasattr(expr, "toarray"):
    expr = expr.toarray()
else:
    expr = np.asarray(expr)
pt = adata_hvg.obs.loc[valid, "pseudotime"].to_numpy()
rho = np.array([spearmanr(pt, expr[:, i]).statistic for i in range(expr.shape[1])])
idx = np.r_[np.argsort(rho)[:12], np.argsort(rho)[-12:]]
bins = pd.qcut(pd.Series(pt).rank(method="first"), 12, labels=False).to_numpy()
hm = pd.DataFrame(expr[:, idx], columns=gene_names[idx]).assign(bin=bins).groupby("bin").mean().T
z = hm.sub(hm.mean(axis=1), axis=0).div(hm.std(axis=1).replace(0, 1), axis=0)
fig, ax = plt.subplots(figsize=(9, 7), dpi=300)
sns.heatmap(z, cmap="vlag", center=0, ax=ax, xticklabels=range(1, 13))
ax.set(title="ModuleDrops: transcriptional programs (0→1→2→4 h trajectory)",
       xlabel="DPT bin (early → late)", ylabel="DPT-associated HVGs")
plt.tight_layout()
plt.show()
pd.DataFrame({"gene": gene_names[idx], "spearman_rho_with_dpt": rho[idx]}).sort_values("spearman_rho_with_dpt").to_csv(
    "cip_trajectory_genes.tsv", sep="\t", index=False
)
#note4fig: fig3d

In [ ]:
import scanpy as sc
adata = sc.read_h5ad("/Users/apple/Downloads/cip/CIP_knee.h5ad")

In [ ]:
# Normalize
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)
adata_full = adata.copy()

sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor="seurat")
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
# PCA
sc.tl.pca(adata_hvg, n_comps=30)
# neighbors
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30)
# UMAP
sc.tl.umap(adata_hvg)
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
sc.pl.umap(adata_hvg, color="time", size=20, ax=ax, show=False, frameon=True)
for collection in ax.collections:
    collection.set_rasterized(True)
plt.tight_layout()
plt.close()

In [ ]:
import numpy as np
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
# Connectivity check
n_comp, labels = connected_components(adata_hvg.obsp["connectivities"], directed=False)
adata_hvg.obs["graph_component"] = labels.astype(str)
print(f"Graph components: {n_comp}")
# Root = 1h (treated)
root_time = "1h"
root_candidates = np.flatnonzero(adata_hvg.obs["time"].astype(str).eq(root_time).to_numpy())
pc_centroid = adata_hvg.obsm["X_pca"][root_candidates].mean(axis=0)
root_idx = root_candidates[np.argmin(((adata_hvg.obsm["X_pca"][root_candidates] - pc_centroid) ** 2).sum(axis=1))]
adata_hvg.uns["iroot"] = int(root_idx)
print(f"DPT root: {adata_hvg.obs_names[root_idx]} (time={root_time})")
# DPT
sc.tl.diffmap(adata_hvg, n_comps=15, random_state=42)
sc.tl.dpt(adata_hvg, n_dcs=10)
adata_hvg.obs["pseudotime"] = adata_hvg.obs["dpt_pseudotime"].replace([np.inf, -np.inf], np.nan)
print(f"Finite pseudotime: {adata_hvg.obs['pseudotime'].notna().sum()}")

In [ ]:
# UMAP: time (left) + pseudotime trajectory (right)
# Same coordinates as original UMAP
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
TIME_ORDER = ["0h","1h","2h","4h"]
TIME_COLORS = {"0h":"#440154","1h":"#31688e","2h":"#35b779","4h":"#fde725"}
fig, axes = plt.subplots(1, 2, figsize=(11, 5), dpi=300)
c = pd.DataFrame(
    adata_hvg.obsm["X_umap"],
    index=adata_hvg.obs_names,
    columns=["UMAP1","UMAP2"]
).join(adata_hvg.obs[["time","pseudotime"]])
# Left: UMAP by sampling time (same as original)
for time in TIME_ORDER:
    d = c[c.time.astype(str).eq(time)]
    axes[0].scatter(d.UMAP1, d.UMAP2, s=10, c=TIME_COLORS[time], alpha=.35, label=time, rasterized=True)
axes[0].set(title="UMAP by sampling time", xlabel="UMAP1", ylabel="UMAP2")
axes[0].legend(frameon=False, markerscale=2)
# Right: UMAP by pseudotime with trajectory arrows
valid = c.pseudotime.notna()
axes[1].scatter(c.loc[~valid, "UMAP1"], c.loc[~valid, "UMAP2"], s=10, c="#d9d9d9", alpha=.25, rasterized=True)
sc_plot = axes[1].scatter(c.loc[valid, "UMAP1"], c.loc[valid, "UMAP2"], s=10, c=c.loc[valid, "pseudotime"], cmap="viridis", alpha=.5, rasterized=True)
d = c.loc[valid].copy()
d["bin"] = pd.qcut(d.pseudotime.rank(method="first"), 12, labels=False)
centers = d.groupby("bin", observed=True)[["UMAP1","UMAP2","pseudotime"]].median().sort_values("pseudotime")
axes[1].plot(centers.UMAP1, centers.UMAP2, c="black", lw=2.2, zorder=5)
for i in range(len(centers)-1):
    axes[1].annotate("", xy=centers.iloc[i+1][["UMAP1","UMAP2"]], xytext=centers.iloc[i][["UMAP1","UMAP2"]],
                     arrowprops={"arrowstyle":"->","color":"black","lw":1.7}, zorder=6)
r = c.iloc[root_idx]
axes[1].scatter(r.UMAP1, r.UMAP2, marker="*", s=200, c="#f9c74f", edgecolor="black", zorder=7)
axes[1].set(title="DPT trajectory from 0h root", xlabel="UMAP1", ylabel="UMAP2")
plt.colorbar(sc_plot, ax=axes[1], shrink=0.6, label="pseudotime")
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3c_part1 (panel2 was extracted and used)

In [ ]:
# UMAP: time (left) + pseudotime trajectory (right)
# Same coordinates as original UMAP
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120)
ax = axes[0]
sc.pl.umap(adata_hvg, color="time", size=20, ax=ax, show=False, frameon=True)
for collection in ax.collections:
    collection.set_rasterized(True)
ax.set_title("Time point")
ax = axes[1]
sc.pl.umap(adata_hvg, color="dpt_pseudotime", size=20, ax=ax, show=False, frameon=True, cmap="viridis")
for collection in ax.collections:
    collection.set_rasterized(True)
ax.set_title("DPT pseudotime")
plt.tight_layout()
plt.show()
#note4fig: fig3a_part1 (panel1 was extracted and used)

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np

files = {
    "0h": "/Users/Apple/Downloads/bacdrops/cip/0h.matrix.tsv",
    "1h": "/Users/Apple/Downloads/bacdrops/cip/1h.matrix.tsv",
    "2h": "/Users/Apple/Downloads/bacdrops/cip/2h.matrix.tsv",
    "4h": "/Users/Apple/Downloads/bacdrops/cip/4h.matrix.tsv",
}
adata_list = []
rrna_genes = {
    "gene-b0201",
    "gene-b0204",
    "gene-b0205",
    "gene-b2588",
    "gene-b2589",
    "gene-b2591",
    "gene-b3272",
    "gene-b3274",
    "gene-b3275",
    "gene-b3278",
    "gene-b3756",
    "gene-b3758",
    "gene-b3759",
    "gene-b3851",
    "gene-b3854",
    "gene-b3855",
    "gene-b3968",
    "gene-b3970",
    "gene-b3971",
    "gene-b4007",
    "gene-b4009",
    "gene-b4010",
}
for tp, fn in files.items():
    df = pd.read_csv(fn, sep="\t", index_col=0)
    ad = sc.AnnData(df.T)
    ad.obs_names = [f"{tp}_{x}" for x in ad.obs_names]
    ad.obs["time"] = tp
    # QC
    ad.obs["total_umis"] = np.asarray(ad.X.sum(axis=1)).ravel()
    ad.obs["n_genes"] = np.asarray((ad.X > 0).sum(axis=1)).ravel()
    rrna_mask = ad.var_names.isin(rrna_genes)
    rrna_counts = np.asarray(ad[:, rrna_mask].X.sum(axis=1)).ravel()
    ad.obs["rrna_pct"] = rrna_counts / ad.obs["total_umis"] * 100
    ad = ad[:, ~rrna_mask].copy()
    adata_list.append(ad)

adata = sc.concat(adata_list, join="outer", label="batch", keys=list(files.keys()), fill_value=0)
print(adata)

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sc.pp.normalize_total(adata, target_sum=10000)
expr = adata.X
if hasattr(expr, "toarray"):
    expr = expr.toarray()
timepoints = ["0h", "1h", "2h", "4h"]
colors = {"0h": "#4C72B0", "1h": "#55A868", "2h": "#C44E52", "4h": "#8172B3"}

fig, ax = plt.subplots(figsize=(3, 3), dpi=300)
summary = []
for tp in timepoints:
    mask = adata.obs["time"] == tp
    X = expr[mask]
    mean = X.mean(axis=0)
    var = X.var(axis=0)
    cv2 = var / (mean**2 + 1e-8)
    # remove extremely low expressed genes
    keep = mean > 0.05
    mean = mean[keep]
    cv2 = cv2[keep]
    log_mean = np.log10(mean)
    log_cv2 = np.log10(cv2)
    bins = np.linspace(log_mean.min(), log_mean.max(), 10)
    x_med = []
    y_med = []
    y_low = []
    y_high = []
    for i in range(len(bins) - 1):
        idx = (log_mean >= bins[i]) & (log_mean < bins[i + 1])
        if idx.sum() < 5:
            continue
        x_med.append(np.median(log_mean[idx]))
        y = log_cv2[idx]
        y_med.append(np.median(y))
        y_low.append(np.percentile(y, 25))
        y_high.append(np.percentile(y, 75))
    x_med = np.array(x_med)
    y_med = np.array(y_med)
    yerr = np.vstack([y_med - np.array(y_low), np.array(y_high) - y_med])
    ax.errorbar(
        x_med, y_med, yerr=yerr, fmt="o", markersize=4, capsize=2, linewidth=0.5, color=colors[tp], label=tp, alpha=0.9
    )
    summary.append({"time": tp, "median_CV2": np.median(cv2), "median_logCV2": np.median(log_cv2), "n_genes": len(cv2)})

ax.set_xlabel("log$_{10}$ Mean normalized expression")
ax.set_ylabel("log$_{10}$ CV$^2$")
ax.set_ylim(-1.5, 3)
ax.set_xlim(-1.5, 2)
ax.legend(frameon=False, fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3e_part1

summary = pd.DataFrame(summary)
print(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4), dpi=300)
time_order = ["0h", "1h", "2h", "4h"]
method_order = ["Knee", "ModuleDrops"]

width = 0.32
positions = []
data = []
colors = {"Knee": "#F58518", "ModuleDrops": "#4C78A8"}
for i, tp in enumerate(time_order):
    for j, method in enumerate(method_order):
        values = df[(df["time"] == tp) & (df["method"] == method)]["log10_CV2"]
        data.append(values)
        if method == "Knee":
            pos = i - width / 2
        else:
            pos = i + width / 2
        positions.append(pos)

bp = ax.boxplot(data, positions=positions, widths=width, patch_artist=True, showfliers=False)

for i, box in enumerate(bp["boxes"]):
    method = method_order[i % 2]
    box.set_facecolor(colors[method])
    box.set_alpha(0.75)
for element in ["whiskers", "caps", "medians"]:
    for line in bp[element]:
        line.set_linewidth(0.8)

ax.set_xticks(np.arange(len(time_order)))
ax.set_xticklabels(["0 h", "1 h", "2 h", "4 h"])
ax.set_ylabel("log$_{10}$(CV$^2$)")
ax.set_xlabel("Ciprofloxacin exposure time")
ax.set_ylim(-2, 4)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors["Knee"], alpha=0.75, label="Knee"),
    Patch(facecolor=colors["ModuleDrops"], alpha=0.75, label="ModuleDrops"),
]
ax.legend(handles=legend_elements, frameon=False, loc="upper left", fontsize=8)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3f

In [ ]:
adata_knee = sc.read_h5ad("CIP_knee.h5ad")

In [ ]:
adata_new = sc.read_h5ad("CIP_new.h5ad")

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

adata_knee_tmp = adata_knee.copy()
adata_module_tmp = adata_new.copy()
adata_knee_tmp.var_names = adata_knee_tmp.var["gene_name"]
adata_module_tmp.var_names = adata_module_tmp.var["gene_name"]
adata_knee_tmp.var_names_make_unique()
adata_module_tmp.var_names_make_unique()
print("Knee tisB:", "tisB" in adata_knee_tmp.var_names)
print("ModuleDrops tisB:", "tisB" in adata_module_tmp.var_names)

gene = "tisB"
timepoints = ["0h", "1h", "2h", "4h"]
results = []
for method, adata in [("Knee", adata_knee_tmp), ("ModuleDrops", adata_module_tmp)]:
    for tp in timepoints:
        mask = adata.obs["time"] == tp
        x = adata[mask, gene].X
        if hasattr(x, "toarray"):
            x = x.toarray()
        x = x.flatten()
        mean = np.mean(x)
        var = np.var(x)
        cv2 = var / (mean**2 + 1e-8)
        results.append({"method": method, "time": tp, "mean": mean, "variance": var, "CV2": cv2})
import pandas as pd
cv_df = pd.DataFrame(results)
print(cv_df)

cv_plot_df = cv_df[cv_df["time"] != "0h"]
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
colors = {"Knee": "#DD8452", "ModuleDrops": "#4C72B0"}
for method in ["Knee", "ModuleDrops"]:
    tmp = cv_plot_df[cv_plot_df["method"] == method]
    ax.plot(tmp["time"], tmp["CV2"], marker="o", linewidth=1, markersize=6, label=method, color=colors[method])
ax.set_yscale("log")
ax.set_ylabel("tisB expression CV$^2$")
ax.set_xlabel("Ciprofloxacin exposure time")
ax.legend(frameon=False, title="")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3g

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
gene = "tisB"
timepoints = ["0h", "1h", "2h", "4h"]
expr_list = []
for method, adata in [("Knee", adata_knee_tmp), ("ModuleDrops", adata_module_tmp)]:
    # already normalized
    for tp in timepoints:
        x = adata[adata.obs["time"] == tp, gene].X
        if hasattr(x, "toarray"):
            x = x.toarray()
        x = x.flatten()
        # log for visualization
        x = np.log1p(x)
        for v in x:
            expr_list.append({"method": method, "time": tp, "expression": v})
expr_df = pd.DataFrame(expr_list)
print(expr_df.head())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), dpi=300)
sns.violinplot(
    data=expr_df[expr_df.time != "0h"],
    x="time",
    y="expression",
    hue="method",
    split=True,
    inner="quartile",
    linewidth=1,
    palette={"Knee": "#DD8452", "ModuleDrops": "#4C72B0"},
    ax=ax,
)
ax.set_xlabel("Ciprofloxacin exposure time")
ax.set_ylabel("log1p normalized tisB expression")
ax.legend(title="", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3h

In [ ]:
threshold = 1
frac_results = []
for method, adata in [("Knee", adata_knee_tmp), ("ModuleDrops", adata_module_tmp)]:
    for tp in timepoints:
        x = adata[adata.obs["time"] == tp, gene].X
        if hasattr(x, "toarray"):
            x = x.toarray()
        x = x.flatten()
        frac = (x > threshold).sum() / len(x)
        frac_results.append({"method": method, "time": tp, "fraction": frac})
frac_df = pd.DataFrame(frac_results)
print(frac_df)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5), dpi=300)
sns.lineplot(
    data=frac_df[frac_df.time != "0h"],
    x="time",
    y="fraction",
    hue="method",
    marker="o",
    linewidth=2,
    markersize=7,
    palette={"Knee": "#DD8452", "ModuleDrops": "#4C72B0"},
    ax=ax,
)
ax.set_ylabel("tisB-positive cells")
ax.set_xlabel("Ciprofloxacin exposure time")
ax.set_ylim(0, 1)
ax.legend(title="", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3i

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np

files = {
    "0h": "/Users/Apple/Downloads/bacdrops/cip/0h.knee.matrix.tsv",
    "1h": "/Users/Apple/Downloads/bacdrops/cip/1h.knee.matrix.tsv",
    "2h": "/Users/Apple/Downloads/bacdrops/cip/2h.knee.matrix.tsv",
    "4h": "/Users/Apple/Downloads/bacdrops/cip/4h.knee.matrix.tsv",
}
adata_list = []
rrna_genes = {
    "gene-b0201",
    "gene-b0204",
    "gene-b0205",
    "gene-b2588",
    "gene-b2589",
    "gene-b2591",
    "gene-b3272",
    "gene-b3274",
    "gene-b3275",
    "gene-b3278",
    "gene-b3756",
    "gene-b3758",
    "gene-b3759",
    "gene-b3851",
    "gene-b3854",
    "gene-b3855",
    "gene-b3968",
    "gene-b3970",
    "gene-b3971",
    "gene-b4007",
    "gene-b4009",
    "gene-b4010",
}
for tp, fn in files.items():
    df = pd.read_csv(fn, sep="\t", index_col=0)
    ad = sc.AnnData(df.T)
    ad.obs_names = [f"{tp}_{x}" for x in ad.obs_names]
    ad.obs["time"] = tp
    # QC
    ad.obs["total_umis"] = np.asarray(ad.X.sum(axis=1)).ravel()
    ad.obs["n_genes"] = np.asarray((ad.X > 0).sum(axis=1)).ravel()
    rrna_mask = ad.var_names.isin(rrna_genes)
    rrna_counts = np.asarray(ad[:, rrna_mask].X.sum(axis=1)).ravel()
    ad.obs["rrna_pct"] = rrna_counts / ad.obs["total_umis"] * 100
    ad = ad[:, ~rrna_mask].copy()
    adata_list.append(ad)

adata = sc.concat(adata_list, join="outer", label="batch", keys=list(files.keys()), fill_value=0)
print(adata)

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sc.pp.normalize_total(adata, target_sum=10000)
expr = adata.X
if hasattr(expr, "toarray"):
    expr = expr.toarray()
timepoints = ["0h", "1h", "2h", "4h"]
colors = {"0h": "#4C72B0", "1h": "#55A868", "2h": "#C44E52", "4h": "#8172B3"}

fig, ax = plt.subplots(figsize=(3, 3), dpi=300)
summary = []
for tp in timepoints:
    mask = adata.obs["time"] == tp
    X = expr[mask]
    mean = X.mean(axis=0)
    var = X.var(axis=0)
    cv2 = var / (mean**2 + 1e-8)
    # remove extremely low expressed genes
    keep = mean > 0.05
    mean = mean[keep]
    cv2 = cv2[keep]
    log_mean = np.log10(mean)
    log_cv2 = np.log10(cv2)
    bins = np.linspace(log_mean.min(), log_mean.max(), 10)
    x_med = []
    y_med = []
    y_low = []
    y_high = []
    for i in range(len(bins) - 1):
        idx = (log_mean >= bins[i]) & (log_mean < bins[i + 1])
        if idx.sum() < 5:
            continue
        x_med.append(np.median(log_mean[idx]))
        y = log_cv2[idx]
        y_med.append(np.median(y))
        y_low.append(np.percentile(y, 25))
        y_high.append(np.percentile(y, 75))
    x_med = np.array(x_med)
    y_med = np.array(y_med)
    yerr = np.vstack([y_med - np.array(y_low), np.array(y_high) - y_med])
    ax.errorbar(
        x_med, y_med, yerr=yerr, fmt="o", markersize=4, capsize=2, linewidth=0.5, color=colors[tp], label=tp, alpha=0.9
    )
    summary.append({"time": tp, "median_CV2": np.median(cv2), "median_logCV2": np.median(log_cv2), "n_genes": len(cv2)})

ax.set_xlabel("log$_{10}$ Mean normalized expression")
ax.set_ylabel("log$_{10}$ CV$^2$")
ax.set_ylim(-1.5, 3)
ax.set_xlim(-1.5, 2)
ax.legend(frameon=False, fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig3e_part1

summary = pd.DataFrame(summary)
print(summary)